# Order Placement Prediction – Fixed Pipeline v3
- LightGBM + XGBoost OOF ensemble
- Optuna tunes inside CV (no leakage)
- High-signal feature engineering
- OOF averaging for stable test predictions
- Outputs probabilities for Kaggle AUC scoring

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "lightgbm", "xgboost", "optuna", "-q"], check=True)
print("Libraries ready.")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LogisticRegression

import lightgbm as lgb
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

pd.set_option("display.max_columns", None)
SEED     = 42
N_FOLDS  = 5
print("All imports OK.")

In [ ]:
train = pd.read_csv("data/train.csv")
test  = pd.read_csv("data/test.csv")

print(f"Train: {train.shape}")
print(f"Test : {test.shape}")
print("\nClass balance:")
print(train["order_placed"].value_counts(normalize=True).round(3))
display(train.isna().sum()[train.isna().sum() > 0].sort_values(ascending=False))

In [ ]:
# ── USER-LEVEL AGGREGATES ─────────────────────────────────────────────────
user_stats = (
    train.groupby("id", as_index=False)
    .agg(
        user_total_sessions=("order_placed", "count"),
        user_order_rate    =("order_placed", "mean"),
        user_avg_cart      =("f11", "mean"),
        user_avg_items     =("f10", "mean"),
    )
)

train = train.merge(user_stats, on="id", how="left")
test  = test.merge(user_stats,  on="id", how="left")

# fill unseen users in test with global averages
test["user_order_rate"].fillna(train["order_placed"].mean(), inplace=True)
test["user_total_sessions"].fillna(1, inplace=True)
test["user_avg_cart"].fillna(train["f11"].mean(), inplace=True)
test["user_avg_items"].fillna(train["f10"].mean(), inplace=True)

print("User aggregates added.")

In [ ]:
TIME_COLS   = ["f3", "f4", "f5"]
SPARSE_COLS = ["f12", "f13", "f14", "f15", "f17"]


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()

    # timestamps
    for col in TIME_COLS:
        data[col] = pd.to_datetime(data[col], utc=True, errors="coerce")

    data["session_length_sec"]         = (data["f4"] - data["f3"]).dt.total_seconds()
    data["action_delay_sec"]           = (data["f5"] - data["f3"]).dt.total_seconds()
    data["time_left_after_action_sec"] = (data["f4"] - data["f5"]).dt.total_seconds()
    data["action_relative_position"]   = (
        data["action_delay_sec"] / data["session_length_sec"].clip(lower=1)
    ).clip(0, 1)

    data["start_hour"]       = data["f3"].dt.hour
    data["start_dayofweek"]  = data["f3"].dt.dayofweek
    data["start_is_weekend"] = data["start_dayofweek"].isin([5, 6]).astype(int)
    data["action_hour"]      = data["f5"].dt.hour

    # timezone → local hour
    def parse_tz_offset(tz_str):
        try:
            return int(str(tz_str).replace("UTC", "").replace("+", "") or 0)
        except Exception:
            return 0

    data["tz_offset"]   = data["f6"].apply(parse_tz_offset)
    data["local_hour"]  = (data["start_hour"] + data["tz_offset"]) % 24
    data["is_meal_time"]= data["local_hour"].isin([7,8,12,13,18,19,20]).astype(int)

    # cart features
    data["cart_value_per_item"]  = data["f11"] / data["f10"].clip(lower=1)
    data["has_items_in_cart"]    = (data["f10"] > 0).astype(int)
    data["offer_ignore_rate"]    = data["f8"] / data["f15"].fillna(0).clip(lower=1)
    data["promo_accepted"]       = (data["f17"] == "accepted").astype(int)
    data["promo_declined"]       = (data["f17"] == "declined").astype(int)
    data["discount_ratio"]       = data["f13"].fillna(0) / data["f14"].fillna(1).clip(lower=1)

    # NEW high-signal features
    data["cart_meets_threshold"]  = (data["f11"].fillna(0) >= data["f14"].fillna(0)).astype(int)
    data["cart_vs_threshold_gap"] = data["f11"].fillna(0) - data["f14"].fillna(0)
    data["discount_net_savings"]  = data["f13"].fillna(0) * data["promo_accepted"]
    data["items_per_minute"]      = data["f10"].fillna(0) / (data["session_length_sec"].clip(lower=60) / 60)
    data["promo_type_x_response"] = data["f12"].fillna("none").astype(str) + "_" + data["f17"].fillna("none").astype(str)
    data["promo_value_density"]   = data["f13"].fillna(0) / data["f14"].fillna(1).clip(lower=1)
    data["active_until_end"]      = (data["time_left_after_action_sec"].fillna(9999) < 60).astype(int)
    data["cart_x_accepted"]       = data["f11"].fillna(0) * data["promo_accepted"]

    # missingness flags
    for col in SPARSE_COLS:
        data[f"{col}_missing"] = data[col].isna().astype(int)

    # session id prefix
    data["f2_prefix"] = data["f2"].astype(str).str[:2]

    # drop raw columns
    data = data.drop(columns=["id", "f2", "f3", "f4", "f5", "f6"], errors="ignore")
    return data


X      = build_features(train.drop(columns=["order_placed"]))
y      = train["order_placed"].copy()
X_test = build_features(test)

numeric_features     = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Features: {X.shape[1]}  (numeric={len(numeric_features)}, cat={len(categorical_features)})")

In [ ]:
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OrdinalEncoder(
                    handle_unknown="use_encoded_value", unknown_value=-1
                )),
            ]),
            categorical_features,
        ),
    ]
)
print("Preprocessor ready.")

In [ ]:
# ── OPTUNA TUNING INSIDE 3-FOLD CV (no leakage) ───────────────────────────
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

def objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 300, 1200),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 31, 300),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "max_depth":         trial.suggest_int("max_depth", 4, 10),
    }
    pipe = Pipeline([
        ("prep", clone(tree_preprocessor)),
        ("model", lgb.LGBMClassifier(
            **params, class_weight="balanced",
            random_state=SEED, n_jobs=-1, verbose=-1,
        )),
    ])
    scores = cross_val_score(pipe, X, y, cv=inner_cv, scoring="roc_auc", n_jobs=1)
    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=80, show_progress_bar=True)

best_lgb_params = study.best_params
print(f"\nBest Optuna AUC: {study.best_value:.4f}")
print("Best params:", best_lgb_params)

In [ ]:
# ── OOF LOOP (LGB + XGB, no CatBoost) ────────────────────────────────────
outer_cv  = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
pos_weight = int((y == 0).sum() / max((y == 1).sum(), 1))

models = {
    "lgb": Pipeline([
        ("prep", clone(tree_preprocessor)),
        ("model", lgb.LGBMClassifier(
            **best_lgb_params, class_weight="balanced",
            random_state=SEED, n_jobs=-1, verbose=-1,
        )),
    ]),
    "xgb": Pipeline([
        ("prep", clone(tree_preprocessor)),
        ("model", xgb.XGBClassifier(
            n_estimators=700, learning_rate=0.03, max_depth=8,
            subsample=0.8, colsample_bytree=0.7,
            scale_pos_weight=pos_weight, eval_metric="auc",
            random_state=SEED, n_jobs=-1, verbosity=0,
            reg_alpha=0.1, reg_lambda=1.0,
        )),
    ]),
}


def run_oof(model_name, model, X_data, X_test_data, y_data, cv):
    oof_preds  = np.zeros(len(y_data))
    test_preds = np.zeros(len(X_test_data))

    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_data, y_data)):
        X_tr  = X_data.iloc[train_idx]
        X_val = X_data.iloc[val_idx]
        y_tr  = y_data.iloc[train_idx]
        y_val = y_data.iloc[val_idx]

        m = clone(model)
        m.fit(X_tr, y_tr)

        oof_preds[val_idx]  = m.predict_proba(X_val)[:, 1]
        test_preds         += m.predict_proba(X_test_data)[:, 1] / cv.n_splits

        fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
        print(f"  {model_name} fold {fold_idx+1}: AUC = {fold_auc:.4f}")

    cv_auc = roc_auc_score(y_data, oof_preds)
    print(f"  {model_name} OOF AUC: {cv_auc:.4f}\n")
    return oof_preds, test_preds, cv_auc


all_oof  = {}
all_test = {}
all_aucs = {}

for name, model in models.items():
    print(f"Running OOF: {name}")
    oof, test_p, auc = run_oof(name, model, X, X_test, y, outer_cv)
    all_oof[name]  = oof
    all_test[name] = test_p
    all_aucs[name] = auc

In [ ]:
# ── META-LEARNER STACKING ─────────────────────────────────────────────────
meta_train = np.column_stack([all_oof[n]  for n in all_oof])
meta_test  = np.column_stack([all_test[n] for n in all_test])

print(f"Meta-feature shape: train={meta_train.shape}, test={meta_test.shape}")

meta_model = lgb.LGBMClassifier(
    n_estimators=200, learning_rate=0.05, num_leaves=15,
    min_child_samples=10, random_state=SEED, n_jobs=-1, verbose=-1,
)

meta_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
meta_scores = cross_val_score(meta_model, meta_train, y, cv=meta_cv, scoring="roc_auc")
print(f"Meta-learner OOF AUC: {meta_scores.mean():.4f} ± {meta_scores.std():.4f}")

meta_model.fit(meta_train, y)
stacked_test_proba = meta_model.predict_proba(meta_test)[:, 1]

In [ ]:
# ── WEIGHTED AVERAGE BLEND ────────────────────────────────────────────────
total_auc     = sum(all_aucs.values())
weighted_test = np.zeros(len(X_test))

for name in all_test:
    w = all_aucs[name] / total_auc
    weighted_test += w * all_test[name]
    print(f"  {name}: weight={w:.3f}  OOF AUC={all_aucs[name]:.4f}")

# 50% stacked + 50% weighted blend
final_test_proba = 0.5 * stacked_test_proba + 0.5 * weighted_test

# OOF summary
print("\n=== OOF AUC Summary ===")
for name, auc in sorted(all_aucs.items(), key=lambda x: -x[1]):
    print(f"  {name:10s}: {auc:.4f}")

weighted_oof = sum((all_aucs[n] / total_auc) * all_oof[n] for n in all_oof)
stacked_oof  = meta_model.predict_proba(meta_train)[:, 1]
final_oof    = 0.5 * stacked_oof + 0.5 * weighted_oof
print(f"  {'final blend':10s}: {roc_auc_score(y, final_oof):.4f}")

In [ ]:
# ── FEATURE IMPORTANCE ────────────────────────────────────────────────────
imp_model = clone(models["lgb"])
imp_model.fit(X, y)

feat_names = (
    numeric_features
    + list(
        imp_model.named_steps["prep"]
        .named_transformers_["cat"]
        .named_steps["encoder"]
        .get_feature_names_out(categorical_features)
    )
)

importance = imp_model.named_steps["model"].feature_importances_
n = min(len(feat_names), len(importance))
imp_df = (
    pd.DataFrame({"feature": feat_names[:n], "importance": importance[:n]})
    .sort_values("importance", ascending=False)
    .head(25)
)

plt.figure(figsize=(9, 7))
sns.barplot(data=imp_df, x="importance", y="feature", palette="viridis")
plt.title("Top-25 Feature Importances (LightGBM)")
plt.tight_layout()
plt.show()

In [ ]:
# ── EXPORT SUBMISSION ─────────────────────────────────────────────────────
submission = pd.DataFrame({
    "id":          test["id"],
    "order_placed": final_test_proba,
})

submission.to_csv("submission_v3.csv", index=False)
print("Saved → submission_v3.csv")
print(f"Rows  : {len(submission)}")
print(f"Range : [{final_test_proba.min():.4f}, {final_test_proba.max():.4f}]")
print(f"Mean  : {final_test_proba.mean():.4f}")
assert submission["order_placed"].between(0, 1).all(), "Probabilities out of range!"
assert submission["id"].nunique() == len(test), "ID mismatch with test set!"
print("All checks passed. Ready to submit to Kaggle.")
display(submission.head(10))